# Model 3: Optuna Tuning, K-Fold CV, and Advanced Features
This notebook builds upon our LightGBM model by introducing robust hyperparameter tuning,
K-Fold Cross Validation, and additional interaction features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import optuna
import sklearn
import os

sklearn.set_config(transform_output="pandas")

In [ ]:
# Load data
train = pd.read_csv('../datasets/train.csv')
test = pd.read_csv('../datasets/test.csv')

X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

In [ ]:
# 1. Imputation
imputer = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

In [ ]:
# 2. Feature Engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_out = X.copy()
        
        denom_screen = X_out['daily_screen_time_hours'].replace(0, 0.001)
        denom_notif = X_out['notifications_per_day'].replace(0, 0.001)
        
        # Original time ratio features
        X_out['social_media_ratio'] = X_out['social_media_hours'] / denom_screen
        X_out['gaming_ratio'] = X_out['gaming_hours'] / denom_screen
        X_out['work_study_ratio'] = X_out['work_study_hours'] / denom_screen
        
        # New advanced features
        X_out['app_opens_per_hour'] = X_out['app_opens_per_day'] / denom_screen
        X_out['notifications_to_opens_ratio'] = X_out['app_opens_per_day'] / denom_notif
        X_out['sleep_deficit'] = 8.0 - X_out['sleep_hours']
        
        return X_out

In [ ]:
# 3. Final Preprocessing (Categorical)
final_preprocessor = ColumnTransformer(
    transformers=[
        # Using OrdinalEncoder for tree-based models
        ('cat_encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [ ]:
# Build Preprocessing Pipeline
preprocessor = Pipeline(steps=[
    ('imputer', imputer),
    ('engineer', FeatureEngineer()),
    ('final_preprocessor', final_preprocessor)
])

print("Preprocessing data...")
X_preprocessed = preprocessor.fit_transform(X)
X_test_preprocessed = preprocessor.transform(X_test)

# Convert categorical columns to category dtype for LightGBM
for col in categorical_cols:
    X_preprocessed[col] = X_preprocessed[col].astype('category')
    X_test_preprocessed[col] = X_test_preprocessed[col].astype('category')

In [ ]:
# 4. Optuna Hyperparameter Tuning
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'verbose': -1
    }
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X_preprocessed, y):
        X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(scores)

print("Starting Optuna tuning (10 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"Best ROC-AUC Score: {study.best_value}")
print(f"Best Params: {study.best_params}")

In [ ]:
# 5. Train Final Model and Predict
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1

final_model = LGBMClassifier(**best_params)
final_model.fit(X_preprocessed, y)

print("Final model trained.")

preds = final_model.predict_proba(X_test_preprocessed)[:, 1]

os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': preds})
submission.to_csv('../submissions/submission_3.csv', index=False)
print("Submission saved to submissions/submission_3.csv")